<a href="https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded:", "Yes" if hf_token else "No")

Token loaded: Yes


In [2]:
!pip install -q huggingface_hub pandas pyarrow

from huggingface_hub import login
login(token=hf_token)

print("Logged in to Hugging Face successfully")

Logged in to Hugging Face successfully


In [3]:
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [4]:
import pandas as pd
from huggingface_hub import hf_hub_download

# Load dim_content (content-level attributes)
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)
dim_content = pd.read_parquet(dim_content_path)

# Load fact table for mid-panel month (March 2026)
fact_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)
fact_march = pd.read_parquet(fact_path)

print("dim_content shape:", dim_content.shape)
print("dim_content columns:", list(dim_content.columns))
print()
print("fact_march shape:", fact_march.shape)
print("fact_march columns:", list(fact_march.columns))

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content shape: (519606, 26)
dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

fact_march shape: (9841378, 30)
fact_march columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chat

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
For my lane (Refresh / Content Opportunity Scoring), one row = one content page, identified by content_hash_id, on a given report_date. I'm using the fact_content_daily_performance table for March 2026 (month=2026-03) as my mid-panel development month, joined with dim_content for page-level attributes (content_type, dates, search_volume, etc.).

Time window: March 1–31, 2026 (mid-panel month, per the assignment's guidance to avoid the sealed final month).

What I'd predict or rank: a priority score per content page, indicating how urgent it is for a reviewer to look at that page — built from trend and performance signals observed within this window.

One thing I deliberately exclude: rows where gsc_data_available or ga4_data_available is False — since those rows don't reflect real observed performance, they'd distort any trend or score built from them.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# check: is content_hash_id + report_date really unique in fact_march? (grain check)
dupes = fact_march.duplicated(subset=["content_hash_id", "report_date"]).sum()
print(f"Duplicate (content_hash_id, report_date) pairs: {dupes}")

# confirm dim_content grain: one row per content_hash_id
dim_dupes = dim_content.duplicated(subset=["content_hash_id"]).sum()
print(f"Duplicate content_hash_id in dim_content: {dim_dupes}")

print(f"\nfact_march date range: {fact_march['report_date'].min()} to {fact_march['report_date'].max()}")


Duplicate (content_hash_id, report_date) pairs: 0
Duplicate content_hash_id in dim_content: 0

fact_march date range: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
**Feature** (known before/at decision time):
- content_type, search_volume, competition, competition_level, cpc, backlinks (from dim_content — page characteristics)
- content_age_days (derived: report_date - content_created_date)
- days_since_last_optimized (derived: report_date - last_optimized_date)
- gsc_impressions, gsc_clicks, gsc_avg_position (recent observed performance, available at decision time)

**Label / proxy**:
- A declining-trend proxy built from comparing gsc_avg_position or gsc_impressions across recent days within the window — NOT a direct column, since no explicit "should_review" label exists in the warehouse.

**Context** (identifiers/reference, not features):
- content_hash_id, client_hash_id, report_date, keyword_hash_id, url_hash_id

**Excluded** (deliberately, with reason):
- is_deleted == True rows — a deleted page isn't a live candidate for review, so including it would pollute the scoring pool.
- rows where gsc_data_available == False — no real signal to score on; including them would introduce fake zeros that look like "no traffic" instead of "no data."
- provider_used, model_used — these describe how content was generated, not how it's performing; irrelevant to a review-priority signal and risk becoming a spurious correlate of client/tooling choice rather than actual page health.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# confirm which planned feature columns actually exist and check basic completeness
feature_cols = ["content_type", "search_volume", "competition", "competition_level", "cpc", "backlinks"]
print("dim_content feature columns present:", [c for c in feature_cols if c in dim_content.columns])
print(dim_content[feature_cols].isna().mean())  # % missing per column


dim_content feature columns present: ['content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'backlinks']
content_type         0.000000
search_volume        0.274481
competition          0.274481
competition_level    0.278011
cpc                  0.274481
backlinks            0.514763
dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Verifying three claims about my slice with real queries: (1) grain — one row really is one content page per day, (2) row count and date span for my lane's slice, (3) availability — filtering with IS TRUE and showing how many rows survive.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: GRAIN — confirm one row = one content page per day
grain_check = fact_march.duplicated(subset=["content_hash_id", "report_date"]).sum()
print(f"Query 1 (grain): duplicate (content_hash_id, report_date) pairs = {grain_check}")
print("-> confirms one row = one content page, one day")

Query 1 (grain): duplicate (content_hash_id, report_date) pairs = 0
-> confirms one row = one content page, one day


In [8]:
# Query 2: ROW COUNT + DATE SPAN for eligible pages in this lane
eligible_march = fact_march[fact_march["content_hash_id"].isin(
    dim_content[dim_content["is_deleted"] == False]["content_hash_id"]
)]

print(f"Query 2: eligible (non-deleted) rows in March slice = {len(eligible_march):,}")
print(f"Unique content pages: {eligible_march['content_hash_id'].nunique():,}")
print(f"Date span: {eligible_march['report_date'].min()} to {eligible_march['report_date'].max()}")

Query 2: eligible (non-deleted) rows in March slice = 9,640,288
Unique content pages: 324,947
Date span: 2026-03-01 to 2026-03-31


In [9]:
# Query 3: AVAILABILITY — filter with IS TRUE, show how many rows survive
available_rows = fact_march[
    (fact_march["gsc_data_available"] == True) &
    (fact_march["ga4_data_available"] == True)
]

total = len(fact_march)
survived = len(available_rows)

print(f"Query 3 (availability): {survived:,} / {total:,} rows have BOTH gsc_data_available IS TRUE and ga4_data_available IS TRUE")
print(f"That's {survived/total*100:.1f}% of the March slice")

Query 3 (availability): 364,347 / 9,841,378 rows have BOTH gsc_data_available IS TRUE and ga4_data_available IS TRUE
That's 3.7% of the March slice


Building a five-feature frame for content-page review priority, using March 2026 data. Each feature is something a reviewer/model would have known at the moment of deciding whether to review a page — no future information.

In [ ]:
import numpy as np

# aggregate March fact data to one row per content page (mean/sum over the month)
page_agg = fact_march.groupby("content_hash_id").agg(
    avg_gsc_position=("gsc_avg_position", "mean"),
    total_gsc_impressions=("gsc_impressions", "sum"),
    total_gsc_clicks=("gsc_clicks", "sum"),
    days_with_data=("gsc_data_available", "sum")
).reset_index()

# merge with dim_content for page-level attributes
features = page_agg.merge(
    dim_content[["content_hash_id", "content_created_date", "search_volume", "backlinks", "is_deleted"]],
    on="content_hash_id", how="left"
)

# feature 1: avg_gsc_position — known because it's the observed ranking position during the window
# feature 2: total_gsc_impressions — known because it's observed traffic volume during the window
# feature 3: total_gsc_clicks — known because it's observed click volume during the window
# feature 4: content_age_days — known because created_date is set at publish time, before the decision
features["content_age_days"] = (
    pd.to_datetime("2026-03-31") - pd.to_datetime(features["content_created_date"])
).dt.days

# feature 5: search_volume — known because it's a keyword-level property, set independent of page performance
features_final = features[[
    "content_hash_id", "avg_gsc_position", "total_gsc_impressions",
    "total_gsc_clicks", "content_age_days", "search_volume"
]]

print(f"Feature frame shape: {features_final.shape}")
features_final.head(10)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.